In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from sqlalchemy import create_engine

In [2]:
username = "root"
password = "jessepinkman"
host = "localhost"
database = "company_db"

In [3]:
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

In [4]:
llm = ChatOllama(model="llama3")

In [5]:
schema = """
Table: employees
Columns: employee_id, employee_name, age, gender, department_id, salary, joining_date, city

Table: departments
Columns: department_id, department_name

Table: customers
Columns: customer_id, customer_name, gender, age, city, phone_number, email, registration_date

Table: products
Columns: product_id, product_name, category, brand, price, stock_quantity

Table: orders
Columns: order_id, customer_id, product_id, employee_id, quantity, order_amount, order_date, payment_method, order_status

Table: sales
Columns: sale_id, employee_id, product_name, sale_amount, sale_date, region
"""

In [7]:
prompt = ChatPromptTemplate.from_template(
    """
    You are an expert MySQL query generator.

    Database Schema:
    {schema}

    User Question:
    {question}

    Rules:
    1. Return only SQL query
    2. Do not explain anything
    3. Use only given tables and columns
    """
)

In [8]:
print(prompt)

input_variables=['question', 'schema'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='\n    You are an expert MySQL query generator.\n\n    Database Schema:\n    {schema}\n\n    User Question:\n    {question}\n\n    Rules:\n    1. Return only SQL query\n    2. Do not explain anything\n    3. Use only given tables and columns\n    '), additional_kwargs={})]


In [9]:
def run_sql_agent(user_question):
    formatted_prompt = prompt.format(
        schema = schema,
        question = user_question
    )

    response = llm.invoke(formatted_prompt)

    generated_sql = response.content.strip()

    print("Generated SQL Query:")
    print(generated_sql)
    print("-" * 50)
    
    try:
        data = pd.read_sql(generated_sql, engine)
        print("result: ")
        print(data)

        chart_prompt = f"""
        User Question: {user_question}

        Data Columns: {list(data.columns)}

        Should this result be shown as a chart?

        Only answer YES or NO.
        """

        chart_response = llm.invoke(chart_prompt)
        needs_chart = chart_response.content.strip().upper()

        if needs_chart == "YES" and len(data.columns) >= 2:

            x_column = data.columns[0]
            y_column = data.columns[1]

            plt.figure(figsize=(8,5))
            plt.bar(data[x_column], data[y_column])
            plt.title(user_question)
            plt.xlabel(x_column)
            plt.ylabel(y_column)
            plt.xticks(rotation=45)
            plt.show()

        else:
            print("No chart needed")
    
    except Exception as e:
        print("Error: ", e)

In [10]:
run_sql_agent("Show sales by region")

Generated SQL Query:
SELECT 
    s.region,
    SUM(s.sale_amount) AS total_sales
FROM 
    sales s
GROUP BY 
    s.region;
--------------------------------------------------
result: 
  region  total_sales
0   West     446200.0
1  North     374100.0
2  South     342600.0
3   East     106750.0


: 